In [22]:
import sys
sys.path.append("..")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.feature_engineering import FeatureEngineer
from src.preprocess          import Preprocessor
from src.data_loader         import DataLoader
from src.model_improved              import ModelTrainerImproved
from sklearn.metrics import accuracy_score


In [2]:
data_loader = DataLoader(path="../data/raw/order_return_data.csv")

In [3]:
data_loader.load()

Data yuklandi: 100000 qator, 14 ustun


,order_item_id,order_date,delivery_date,item_id,item_size,item_color,brand_id,item_price,user_id,user_title,user_dob,user_state,user_reg_date,return
0,1,2016-06-22,2016-06-27,643,38,navy,30,49.90,30822,Mrs,1969-04-17,Saxony,2016-06-23,0
1,2,2016-06-22,NaN,337,152,grey,30,19.95,30822,Mrs,1969-04-17,Saxony,2016-06-23,0
2,3,2016-06-22,2016-06-27,270,xxl,grey,49,79.90,30823,Mrs,1970-04-22,Baden-Wuerttemberg,2015-03-15,1
3,4,2016-06-22,2016-06-27,142,xxl,grey,49,99.90,30823,Mrs,1970-04-22,Baden-Wuerttemberg,2015-03-15,0
4,5,2016-06-22,2016-06-27,561,xxl,grey,3,14.90,30823,Mrs,1970-04-22,Baden-Wuerttemberg,2015-03-15,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99995,99996,2016-09-11,2016-09-12,156,20,blue,34,29.90,713,Mrs,1959-03-21,Rhineland-Palatinate,2015-02-17,0
99996,99997,2016-09-11,2016-09-12,156,20,brown,34,29.90,713,Mrs,1959-03-21,Rhineland-Palatinate,2015-02-17,0
99997,99998,2016-09-11,1994-12-31,1832,xxl,black,37,26.90,47794,Mr,1985-03-01,Hamburg,2016-09-10,0
99998,99999,2016-09-11,1994-12-31,1832,xxl,black,37,26.90,47794,Mr,1985-03-01,Hamburg,2016-09-10,0


In [4]:
x_train, x_test, y_train, y_test = data_loader.split(target_column="return")

Train: 80000, Test: 20000


In [5]:
prep = Preprocessor()

In [6]:
x_train_filled, fill_values = prep.fillna_train(df=x_train)
x_test_filled = prep.fillna_test(df=x_test, fill_values=fill_values)

/Users/plosx/Desktop/machine-learning/modular_programming/order_return_prediction/notebooks/../src/preprocess.py:19: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(fill_values[col], inplace=True)
/Users/plosx/Desktop/machine-learning/modular_programming/order_return_prediction/notebooks/../src/preprocess.py:16: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work

In [7]:
fe = FeatureEngineer()

In [8]:
x_train_fe = fe.add_features(x_train_filled)
x_test_fe = fe.add_features(x_test_filled)

✓ Yangi featurelar qo'shildi: 17 ustun
✓ Yangi featurelar qo'shildi: 17 ustun


In [9]:
x_train_encoded, encoders, one_hot_cols = prep.encode_train(df=x_train_fe)
x_test_encoded = prep.encode_test(
  df=x_test_fe, 
  encoders=encoders, 
  onehot_cols=one_hot_cols, 
  train_columns=x_train_fe.columns
)

In [10]:
x_train_scaled, scalers = prep.scale_train(df=x_train_encoded)
x_test_scaled = prep.scale_test(df=x_test_encoded, scalers=scalers)

In [11]:
skewed_features = fe.fix_skewness(df=x_train_scaled)

In [12]:
skewed_features

['brand_id', 'item_price', 'is_late_delivery', 'user_age']

In [13]:
x_train_scaled.info()

<class 'pandas.core.frame.DataFrame'>
Index: 80000 entries, 75220 to 15795
Data columns (total 17 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   order_item_id     80000 non-null  float64
 1   order_date        80000 non-null  float64
 2   delivery_date     80000 non-null  float64
 3   item_id           80000 non-null  float64
 4   item_size         80000 non-null  float64
 5   item_color        80000 non-null  float64
 6   brand_id          80000 non-null  float64
 7   item_price        80000 non-null  float64
 8   user_id           80000 non-null  float64
 9   user_title        80000 non-null  float64
 10  user_dob          80000 non-null  float64
 11  user_state        80000 non-null  float64
 12  user_reg_date     80000 non-null  float64
 13  delivery_days     80000 non-null  float64
 14  is_fast_delivery  80000 non-null  float64
 15  is_late_delivery  80000 non-null  float64
 16  user_age          80000 non-null  float64

In [14]:
X_train_transformed = x_train_scaled.copy()
X_test_transformed = x_test_scaled.copy()

for col in skewed_features:
    if (x_train_fe[col] >= 0).all():
        x_train_fe[col] = np.log1p(x_train_fe[col])
        x_test_fe[col] = np.log1p(x_test_fe[col])

In [15]:
X_train_transformed

,order_item_id,order_date,delivery_date,item_id,item_size,item_color,brand_id,item_price,user_id,user_title,user_dob,user_state,user_reg_date,delivery_days,is_fast_delivery,is_late_delivery,user_age
75220,0.872828,0.862590,0.124645,1.569487,-0.702821,0.382549,-1.033869,0.305712,1.187301,0.193858,0.824885,0.171045,-1.137186,0.098315,0.812547,-0.447489,-0.824101
48955,-0.036957,0.137221,0.106129,0.826573,0.872707,0.382549,2.546649,-0.317006,0.127402,0.193858,0.611124,0.638453,0.462274,0.102021,-1.230697,-0.447489,-0.632924
44966,-0.175131,-0.064270,0.099957,0.856460,0.780029,-1.275924,0.632312,1.759756,0.025619,0.193858,-0.114668,-0.296363,-1.052060,0.102021,-1.230697,-0.447489,0.131784
13568,-1.262716,-1.232919,0.062925,-1.106143,-0.517465,0.382549,-1.033869,-0.317006,-1.180691,0.193858,-0.792972,0.638453,-1.137186,0.100786,0.812547,-0.447489,0.800903
92727,1.479247,1.467063,0.064159,1.254958,0.872707,0.337725,2.830254,-1.001995,1.055979,0.193858,-0.011189,0.171045,0.619084,0.019239,0.812547,-0.447489,-0.011599
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6265,-1.515682,-1.313516,0.057987,-0.697682,0.826368,1.144550,-0.076701,-0.109433,0.344919,0.193858,0.313639,0.171045,-1.137186,0.098315,0.812547,-0.447489,-0.298364
54886,0.168485,0.258116,0.108597,1.696152,0.362977,-0.917335,1.199523,0.515360,0.923020,0.193858,1.422866,-1.231178,-0.012635,0.100786,0.812547,-0.447489,-1.445425
76820,0.928250,0.902888,0.127113,1.616453,-0.239430,-0.738041,-1.033869,1.966291,1.206590,0.193858,0.334570,0.638453,0.529479,0.099550,0.812547,-0.447489,-0.346159
860,-1.702904,-1.434411,0.056753,-1.313931,-0.610143,0.696314,-1.033869,1.136001,0.273101,0.193858,0.565991,-0.062659,1.067112,0.100786,0.812547,-0.447489,-0.585130


In [16]:
x_train_lasso_selected, x_test_lasso_selected = fe.lasso_selector(X_train_transformed, X_test_transformed, y_train)

Selected features for Linear Models: ['order_item_id', 'order_date', 'delivery_date', 'item_id', 'item_size', 'item_color', 'brand_id', 'item_price', 'user_id', 'user_title', 'user_dob', 'user_state', 'user_reg_date', 'is_fast_delivery', 'is_late_delivery', 'user_age']


In [17]:
trainer = ModelTrainerImproved()

trainer.logReg(x_train_lasso_selected, y_train)

Logistic Regression o'rgatildi


{'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42)}

In [18]:
x_train_tree_selected, x_test_tree_selected = fe.random_forest_selector(X_train_transformed, X_test_transformed, y_train)

Top features for Tree Models: ['order_item_id', 'user_id', 'item_id', 'user_dob', 'item_price', 'delivery_date', 'delivery_days', 'item_size', 'item_color', 'user_reg_date']


In [20]:
models = trainer.RandomForest(x_train_tree_selected, y_train)


Logistic Regression o'rgatildi


In [23]:
acc_logReg = accuracy_score(y_test, models["Logistic Regression"].predict(x_test_lasso_selected))

In [26]:
acc_rf = accuracy_score(y_test, models["Random Forest"].predict(x_test_tree_selected))

In [27]:
acc_logReg

0.5763

In [28]:
acc_rf

0.65585